In [ ]:
!pip install google-api-python-client pandas

In [ ]:
import pandas as pd
from googleapiclient.discovery import build

# Your API key
API_KEY = "API kEY IS Remove by USER" # api key is remive by user 

# Enter the ID of a political news video here
# Example URL: https://youtu.be/o4uO88_L1C4?si=pHDHmD8_7f8veWih
# The ID would be: o4uO88_L1C4
VIDEO_ID = "o4uO88_L1C4"

def get_all_youtube_comments(video_id, api_key):
    # Build the API service
    youtube = build("youtube", "v3", developerKey=api_key)
    comments_list = []

    # Initial request
    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        maxResults=100,
        textFormat="plainText"
    )

    print("Fetching comments... Please wait.")

    # Loop until all pages (comments) are fetched
    while request:
        try:
            response = request.execute()

            for item in response.get('items', []):
                comment = item['snippet']['topLevelComment']['snippet']
                comments_list.append({
                    'Author': comment.get('authorDisplayName', 'Unknown'),
                    'Comment_Text': comment.get('textDisplay', ''),
                    'Likes': int(comment.get('likeCount', 0)),
                    'Published_At': comment.get('publishedAt', '')
                })

            # If there's a next page, get data for the next page
            if 'nextPageToken' in response:
                request = youtube.commentThreads().list(
                    part="snippet",
                    videoId=video_id,
                    pageToken=response['nextPageToken'],
                    maxResults=100,
                    textFormat="plainText"
                )
            else:
                request = None # End loop

        except Exception as e:
            print(f"An error occurred: {e}")
            break

    # Convert data to a Pandas DataFrame
    df = pd.DataFrame(comments_list)
    return df

# Run the function
df_comments = get_all_youtube_comments(VIDEO_ID, API_KEY)

# Display results
if not df_comments.empty:
    print(f"\nTotal Comments Extracted: {len(df_comments)}")
    display(df_comments.head()) # Display table nicely in Colab

    # Save data to CSV
    df_comments.to_csv('political_news_comments.csv', index=False)
    print("\nData successfully saved to 'political_news_comments.csv'!")
else:
    print("\nNo comments extracted. Please check the Video ID.")

Fetching comments... Please wait.

Total Comments Extracted: 218


,Author,Comment_Text,Likes,Published_At
0,@DyCSTEFOIS,Railmantri jayenge,0,2026-07-04T17:12:23Z
1,@shantpriyachandra3666,We feel that aap dallaa giri achi kar lete hai.,0,2026-07-03T01:24:49Z
2,@ArupBhandari-o4b,Shaktikanta das is a history scholar.. note an...,0,2026-07-01T11:47:43Z
3,@kuldeepsingh-xl4zo,Aajkal vipaksh ki taraf jhukaav sa badh gaya h...,0,2026-07-01T10:13:16Z
4,@Bhupender002singh,Dharmendra pradhan and Nitin Gadkari ko hatao,0,2026-07-01T07:15:15Z



Data successfully saved to 'political_news_comments.csv'!


In [ ]:
"""
YouTube Comments Extractor - using YouTube Data API v3
--------------------------------------------------------

"""

import csv
import time
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# ------------- CONFIG -------------
API_KEY = "API kEY IS Remove by USER"          # <-- paste your API key here
VIDEO_ID = "o4uO88_L1C4"            # <-- paste the video ID (from youtube.com/watch?v=XXXX)
OUTPUT_FILE = "comments.csv"
# -----------------------------------


def get_comments(youtube, video_id):
    """Fetch all comment threads (top-level comments + their replies) for a video."""
    comments = []
    next_page_token = None
    page_count = 0

    while True:
        try:
            request = youtube.commentThreads().list(
                part="snippet,replies",
                videoId=video_id,
                maxResults=100,
                pageToken=next_page_token,
                textFormat="plainText",
                order="relevance"   # or "time" for chronological order
            )
            response = request.execute()
        except HttpError as e:
            if e.resp.status == 403:
                print("Comments are disabled for this video, or quota exceeded.")
            else:
                print(f"An API error occurred: {e}")
            break

        for item in response.get("items", []):
            top_comment = item["snippet"]["topLevelComment"]["snippet"]
            comments.append({
                "comment_id": item["snippet"]["topLevelComment"]["id"],
                "type": "top_level",
                "parent_id": "",
                "author": top_comment.get("authorDisplayName", ""),
                "text": top_comment.get("textOriginal", ""),
                "likes": top_comment.get("likeCount", 0),
                "published_at": top_comment.get("publishedAt", ""),
            })

            # Extract replies if present in the same response (up to 5 shown by default)
            if "replies" in item:
                for reply in item["replies"]["comments"]:
                    r_snippet = reply["snippet"]
                    comments.append({
                        "comment_id": reply["id"],
                        "type": "reply",
                        "parent_id": item["snippet"]["topLevelComment"]["id"],
                        "author": r_snippet.get("authorDisplayName", ""),
                        "text": r_snippet.get("textOriginal", ""),
                        "likes": r_snippet.get("likeCount", 0),
                        "published_at": r_snippet.get("publishedAt", ""),
                    })

        page_count += 1
        print(f"Fetched page {page_count} — total comments so far: {len(comments)}")

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

        time.sleep(0.2)  # small delay to be gentle on the API

    return comments

def save_to_csv(comments, filename):
    if not comments:
        print("No comments to save.")
        return

    keys = comments[0].keys()
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(comments)

    print(f"\nSaved {len(comments)} comments to {filename}")

def main():
    youtube = build("youtube", "v3", developerKey=API_KEY)
    print(f"Fetching comments for video: {VIDEO_ID} ...\n")

    comments = get_comments(youtube, VIDEO_ID)
    save_to_csv(comments, OUTPUT_FILE)


if __name__ == "__main__":
    main()

Fetching comments for video: o4uO88_L1C4 ...

Fetched page 1 — total comments so far: 117
Fetched page 2 — total comments so far: 228
Fetched page 3 — total comments so far: 233

Saved 233 comments to comments.csv


# **API**

In [ ]:
"""
YouTube Comments Extractor - using YouTube Data API v3
--------------------------------------------------------
Extracts all top-level comments + replies from a given video
and saves them into a CSV file for further analysis.
"""

import csv
import os
import time
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# ------------- CONFIG -------------
API_KEY = os.environ.get("YOUTUBE_API_KEY", "API kEY IS Remove by USER")
VIDEO_ID = "yCX5rj9Tmxs"            # from https://www.youtube.com/watch?v=yCX5rj9Tmxs
OUTPUT_FILE = "comments.csv"
# -----------------------------------


def get_all_replies(youtube, parent_id):
    """Fetch ALL replies under a top-level comment (beyond the 5 shown inline)."""
    replies = []
    next_page_token = None

    while True:
        try:
            request = youtube.comments().list(
                part="snippet",
                parentId=parent_id,
                maxResults=100,
                pageToken=next_page_token,
                textFormat="plainText",
            )
            response = request.execute()
        except HttpError as e:
            print(f"Error fetching replies for {parent_id}: {e}")
            break

        for item in response.get("items", []):
            r_snippet = item["snippet"]
            replies.append({
                "comment_id": item["id"],
                "type": "reply",
                "parent_id": parent_id,
                "author": r_snippet.get("authorDisplayName", ""),
                "text": r_snippet.get("textOriginal", ""),
                "likes": r_snippet.get("likeCount", 0),
                "published_at": r_snippet.get("publishedAt", ""),
            })

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break
        time.sleep(0.2)

    return replies


def get_comments(youtube, video_id):
    """Fetch ALL comment threads (top-level comments + every reply) for a video."""
    comments = []
    next_page_token = None
    page_count = 0

    while True:
        try:
            request = youtube.commentThreads().list(
                part="snippet,replies",
                videoId=video_id,
                maxResults=100,
                pageToken=next_page_token,
                textFormat="plainText",
                order="time"   # chronological, safer for exhaustive pulls than "relevance"
            )
            response = request.execute()
        except HttpError as e:
            if e.resp.status == 403:
                print("Comments are disabled for this video, or quota exceeded.")
            else:
                print(f"An API error occurred: {e}")
            break

        for item in response.get("items", []):
            top_snippet = item["snippet"]
            top_comment = top_snippet["topLevelComment"]["snippet"]
            top_id = top_snippet["topLevelComment"]["id"]
            total_reply_count = top_snippet.get("totalReplyCount", 0)

            comments.append({
                "comment_id": top_id,
                "type": "top_level",
                "parent_id": "",
                "author": top_comment.get("authorDisplayName", ""),
                "text": top_comment.get("textOriginal", ""),
                "likes": top_comment.get("likeCount", 0),
                "published_at": top_comment.get("publishedAt", ""),
            })

            inline_replies = item.get("replies", {}).get("comments", [])

            if total_reply_count > len(inline_replies):
                # More replies exist than YouTube inlined here — fetch them all separately.
                full_replies = get_all_replies(youtube, top_id)
                comments.extend(full_replies)
            else:
                for reply in inline_replies:
                    r_snippet = reply["snippet"]
                    comments.append({
                        "comment_id": reply["id"],
                        "type": "reply",
                        "parent_id": top_id,
                        "author": r_snippet.get("authorDisplayName", ""),
                        "text": r_snippet.get("textOriginal", ""),
                        "likes": r_snippet.get("likeCount", 0),
                        "published_at": r_snippet.get("publishedAt", ""),
                    })

        page_count += 1
        print(f"Fetched page {page_count} — total comments so far: {len(comments)}")

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

        time.sleep(0.2)  # small delay to be gentle on the API

    return comments

def save_to_csv(comments, filename):
    if not comments:
        print("No comments to save.")
        return

    keys = comments[0].keys()
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(comments)

    print(f"\nSaved {len(comments)} comments to {filename}")

def main():
    if API_KEY == "PASTE_YOUR_KEY_HERE":
        raise SystemExit("Set YOUTUBE_API_KEY env variable or edit API_KEY in the script.")

    youtube = build("youtube", "v3", developerKey=API_KEY)
    print(f"Fetching comments for video: {VIDEO_ID} ...\n")

    comments = get_comments(youtube, VIDEO_ID)
    save_to_csv(comments, OUTPUT_FILE)


if __name__ == "__main__":
    main()

Fetching comments for video: yCX5rj9Tmxs ...

Fetched page 1 — total comments so far: 100
Fetched page 2 — total comments so far: 200
Fetched page 3 — total comments so far: 300
Fetched page 4 — total comments so far: 400
Fetched page 5 — total comments so far: 502
Fetched page 6 — total comments so far: 605
Fetched page 7 — total comments so far: 709
Fetched page 8 — total comments so far: 817
Fetched page 9 — total comments so far: 952
Fetched page 10 — total comments so far: 1075
Fetched page 11 — total comments so far: 1182
Fetched page 12 — total comments so far: 1305
Fetched page 13 — total comments so far: 1414
Fetched page 14 — total comments so far: 1523
Fetched page 15 — total comments so far: 1626
Fetched page 16 — total comments so far: 1727
Fetched page 17 — total comments so far: 1840
Fetched page 18 — total comments so far: 2149
Fetched page 19 — total comments so far: 2255
Fetched page 20 — total comments so far: 2363
Fetched page 21 — total comments so far: 2464
Fetche

In [ ]:
%%time
youtube = build("youtube", "v3", developerKey=API_KEY)
comments = get_comments(youtube, VIDEO_ID)
save_to_csv(comments, OUTPUT_FILE)

Fetched page 1 — total comments so far: 100
Fetched page 2 — total comments so far: 200
Fetched page 3 — total comments so far: 300
Fetched page 4 — total comments so far: 400
Fetched page 5 — total comments so far: 502
Fetched page 6 — total comments so far: 605
Fetched page 7 — total comments so far: 709
Fetched page 8 — total comments so far: 817
Fetched page 9 — total comments so far: 952
Fetched page 10 — total comments so far: 1075
Fetched page 11 — total comments so far: 1182
Fetched page 12 — total comments so far: 1305
Fetched page 13 — total comments so far: 1414
Fetched page 14 — total comments so far: 1523
Fetched page 15 — total comments so far: 1626
Fetched page 16 — total comments so far: 1727
Fetched page 17 — total comments so far: 1840
Fetched page 18 — total comments so far: 2149
Fetched page 19 — total comments so far: 2255
Fetched page 20 — total comments so far: 2363
Fetched page 21 — total comments so far: 2464
Fetched page 22 — total comments so far: 2565
Fetche

# YouTube Comments Extractor - using `yt-dlp`

In [ ]:
!pip -q install -U yt-dlp pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 74.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.


In [ ]:
!pip install -q -U yt-dlp

In [ ]:
import csv
import yt_dlp

VIDEO_URL = "API kEY IS Remove by USER"
OUTPUT_FILE = "comments_ytdlp.csv"

def get_comments_ytdlp(video_url):
    ydl_opts = {
        "skip_download": True,
        "getcomments": True,
        "extractor_args": {
            "youtube": {
                "max_comments": ["all", "all", "all", "all"],
                "comment_sort": ["top"],
            }
        },
        "quiet": True,
        "no_warnings": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=False)
    raw_comments = info.get("comments", []) or []
    comments = []
    for c in raw_comments:
        comments.append({
            "comment_id": c.get("id", ""),
            "type": "reply" if c.get("parent") not in (None, "root") else "top_level",
            "parent_id": "" if c.get("parent") in (None, "root") else c.get("parent"),
            "author": c.get("author", ""),
            "text": c.get("text", ""),
            "likes": c.get("like_count", 0) or 0,
            "published_at": c.get("timestamp", ""),
        })
    print(f"Total comments fetched: {len(comments)}")
    return comments

def save_to_csv(comments, filename):
    if not comments:
        print("No comments to save.")
        return
    keys = comments[0].keys()
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(comments)
    print(f"Saved {len(comments)} comments to {filename}")

In [ ]:
%%time
comments = get_comments_ytdlp(VIDEO_URL)
save_to_csv(comments, OUTPUT_FILE)

Total comments fetched: 1544
Saved 1544 comments to comments_ytdlp.csv
CPU times: user 3.54 s, sys: 36.5 ms, total: 3.58 s
Wall time: 40.3 s


# YouTube Comments Extractor - using Selenium / Playwright

In [ ]:
!pip install playwright
!playwright install chromium
!playwright install-deps chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 22.8 MB/s eta 0:00:00
184.3 MiB [] 0% 0.0s184.3 MiB [] 0% 25.8s184.3 MiB [] 0% 14.2s184.3 MiB [] 0% 13.2s184.3 MiB [] 1% 6.6s184.3 MiB [] 2% 3.9s184.3 MiB [] 2% 3.4s184.3 MiB [] 3% 2.9s184.3 MiB [] 4% 2.7s184.3 MiB [] 5% 2.5s184.3 MiB [] 6% 2.3s184.3 MiB [] 6% 2.6s184.3 MiB [] 7% 2.7s184.3 MiB [] 7% 2.8s184.3 MiB [] 7% 2.9s184.3 MiB [] 8% 2.7s184.3 MiB [] 9% 2.6s184.3 MiB [] 10% 2.4s184.3 MiB [] 11% 2.3s184.3 MiB [] 12% 2.2s184.3 MiB [] 13% 2.2s184.3 MiB [] 14% 2.2s184.3 MiB [] 15% 2.1s184.3 MiB [] 16% 2.0s184.3 MiB [] 17% 2.0s184.3 MiB [] 18% 1.9s184.3 MiB [] 19% 1.9s184.3 MiB [] 20% 2.0s184.3 MiB [] 20% 2.1s184.3 MiB [] 21% 2.0s184.3 MiB [] 22% 2.0s184.3 MiB [] 24% 1.9s184.3 MiB [] 25% 1.8s184.3 MiB [] 25% 1.9s184.3 MiB [] 26% 1.9s184.3 MiB [] 28% 1.8s184.3 MiB [] 29% 1.7s184.3 MiB [] 30% 1.7s184.3 MiB [] 32% 1.6s184.3 MiB [] 33% 1.5s184.3 MiB [] 35% 1.4s184.3 MiB [] 36% 1.4s184.3 MiB [] 37% 1.3s184.3 MiB [] 39% 1.3s184.3 MiB [

In [ ]:
# Setup (async version, required for Colab's event loop)
# ============================================================
import csv
import time
import asyncio
from playwright.async_api import async_playwright

VIDEO_URL = "API kEY IS Remove by USER"
OUTPUT_FILE = "comments_playwright.csv"
EXPAND_REPLIES = False
SCROLL_PAUSE = 1.5
MAX_STALE_SCROLLS = 5

COMMENT_RENDERER = "ytd-comment-view-model, ytd-comment-renderer"
AUTHOR = "#author-text"
TEXT = "#content-text"
LIKES = "#vote-count-middle"
TIME_LINK = ".published-time-text a"
REPLY_BUTTON = "#more-replies ytd-button-renderer button"


async def _extract_comment_data(comment_el):
    async def safe_text(selector):
        el = await comment_el.query_selector(selector)
        if el:
            txt = await el.inner_text()
            return txt.strip()
        return ""

    return {
        "author": await safe_text(AUTHOR),
        "text": await safe_text(TEXT),
        "likes": (await safe_text(LIKES)) or "0",
        "published_at": await safe_text(TIME_LINK),
    }


async def get_comments_playwright(video_url, max_comments=None):
    comments = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(video_url, wait_until="domcontentloaded")

        try:
            await page.click("text=Accept all", timeout=3000)
        except Exception:
            pass

        await page.mouse.wheel(0, 800)
        await page.wait_for_timeout(2000)

        stale_scrolls = 0
        last_count = 0

        while True:
            if EXPAND_REPLIES:
                for btn in await page.query_selector_all(REPLY_BUTTON):
                    try:
                        await btn.click(timeout=1000)
                        await page.wait_for_timeout(300)
                    except Exception:
                        pass

            threads = await page.query_selector_all(COMMENT_RENDERER)
            current_count = len(threads)
            print(f"Loaded so far: {current_count} comment elements")

            if max_comments and current_count >= max_comments:
                break

            if current_count == last_count:
                stale_scrolls += 1
                if stale_scrolls >= MAX_STALE_SCROLLS:
                    print("No new comments after several scrolls — assuming end reached.")
                    break
            else:
                stale_scrolls = 0

            last_count = current_count

            await page.mouse.wheel(0, 3000)
            await page.wait_for_timeout(int(SCROLL_PAUSE * 1000))

        threads = await page.query_selector_all(COMMENT_RENDERER)
        for el in threads:
            data = await _extract_comment_data(el)
            if data["text"]:
                comments.append(data)

        await browser.close()

    if max_comments:
        comments = comments[:max_comments]

    print(f"Total comments extracted: {len(comments)}")
    return comments

def save_to_csv(comments, filename):
    if not comments:
        print("No comments to save.")
        return
    keys = comments[0].keys()
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(comments)
    print(f"Saved {len(comments)} comments to {filename}")

# YouTube Comments Extractor - using `youtube-comment-downloader`

In [ ]:
!pip install youtube-comment-downloader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 8.0 MB/s eta 0:00:00


In [ ]:
import csv
from youtube_comment_downloader import YoutubeCommentDownloader, SORT_BY_RECENT

VIDEO_URL = "https://www.youtube.com/watch?v=yCX5rj9Tmxs"
OUTPUT_FILE = "comments_downloader.csv"

def get_comments(video_url, max_comments=None):
    downloader = YoutubeCommentDownloader()
    comments = []
    for comment in downloader.get_comments_from_url(video_url, sort_by=SORT_BY_RECENT):
        comments.append({
            "comment_id": comment.get("cid", ""),
            "type": "reply" if comment.get("reply") else "top_level",
            "author": comment.get("author", ""),
            "text": comment.get("text", ""),
            "likes": comment.get("votes", "0"),
            "published_at": comment.get("time", ""),
            "heart": comment.get("heart", False),
        })
        if len(comments) % 500 == 0:
            print(f"Fetched so far: {len(comments)}")
        if max_comments and len(comments) >= max_comments:
            break
    print(f"Total comments fetched: {len(comments)}")
    return comments

def save_to_csv(comments, filename):
    if not comments:
        print("No comments to save.")
        return
    keys = comments[0].keys()
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(comments)
    print(f"Saved {len(comments)} comments to {filename}")

In [ ]:
%%time
comments = get_comments(VIDEO_URL)


Fetched so far: 500
Fetched so far: 1000
Fetched so far: 1500
Fetched so far: 2000
Fetched so far: 2500
Fetched so far: 3000
Fetched so far: 3500
Fetched so far: 4000
Fetched so far: 4500
Fetched so far: 5000
Fetched so far: 5500
Fetched so far: 6000
Fetched so far: 6500
Fetched so far: 7000
Fetched so far: 7500
Fetched so far: 8000
Fetched so far: 8500
Fetched so far: 9000
Fetched so far: 9500
Fetched so far: 10000
Fetched so far: 10500
Total comments fetched: 10843
CPU times: user 30.8 s, sys: 162 ms, total: 30.9 s
Wall time: 3min 46s


# YouTube Comments Extractor - using `yt-dlp` (Updated Script)

In [ ]:
"""
YouTube Comments Extractor - using yt-dlp
-------------------------------------------
"""

import csv
import yt_dlp

VIDEO_URL = "https://www.youtube.com/watch?v=yCX5rj9Tmxs"
OUTPUT_FILE = "comments_ytdlp.csv"


def get_comments_ytdlp(video_url):
    """Fetch all comments for a video using yt-dlp. Returns a list of dicts."""
    ydl_opts = {
        "skip_download": True,
        "getcomments": True,
        "extractor_args": {
            "youtube": {
                # pull every comment thread and every reply, not just the top ones
                "max_comments": ["all", "all", "all", "all"],
                "comment_sort": ["new"],  # chronological — "top" caps out early (YouTube limitation)
            }
        },
        "quiet": True,
        "no_warnings": True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=False)

    raw_comments = info.get("comments", []) or []

    comments = []
    for c in raw_comments:
        comments.append({
            "comment_id": c.get("id", ""),
            "type": "reply" if c.get("parent") not in (None, "root") else "top_level",
            "parent_id": "" if c.get("parent") in (None, "root") else c.get("parent"),
            "author": c.get("author", ""),
            "text": c.get("text", ""),
            "likes": c.get("like_count", 0) or 0,
            "published_at": c.get("timestamp", ""),
        })

    print(f"Total comments fetched: {len(comments)}")
    return comments


def save_to_csv(comments, filename):
    if not comments:
        print("No comments to save.")
        return

    keys = comments[0].keys()
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(comments)

    print(f"Saved {len(comments)} comments to {filename}")


def main():
    comments = get_comments_ytdlp(VIDEO_URL)



if __name__ == "__main__":
    main()

Total comments fetched: 10843
Saved 10843 comments to comments_ytdlp.csv


In [ ]:
"""
YouTube Comments Extractor - using yt-dlp
-------------------------------------------
"""

import csv
import time
import yt_dlp

VIDEO_URL = "https://www.youtube.com/watch?v=yCX5rj9Tmxs"
OUTPUT_FILE = "comments_ytdlp.csv"


def get_comments_ytdlp(video_url):
    """Fetch all comments for a video using yt-dlp."""

    start_fetch = time.perf_counter()

    ydl_opts = {
        "skip_download": True,
        "getcomments": True,
        "extractor_args": {
            "youtube": {
                "max_comments": ["all", "all", "all", "all"],
                "comment_sort": ["new"],
            }
        },
        "quiet": True,
        "no_warnings": True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=False)

    fetch_time = time.perf_counter() - start_fetch

    raw_comments = info.get("comments", []) or []

    comments = []
    for c in raw_comments:
        comments.append({
            "comment_id": c.get("id", ""),
            "type": "reply" if c.get("parent") not in (None, "root") else "top_level",
            "parent_id": "" if c.get("parent") in (None, "root") else c.get("parent"),
            "author": c.get("author", ""),
            "text": c.get("text", ""),
            "likes": c.get("like_count", 0) or 0,
            "published_at": c.get("timestamp", ""),
        })

    print(f"✅ Total comments fetched: {len(comments)}")
    print(f"⏱️ Fetch time: {fetch_time:.2f} seconds")

    return comments, fetch_time


def save_to_csv(comments, filename):
    if not comments:
        print("No comments to save.")
        return

    start_save = time.perf_counter()

    keys = comments[0].keys()
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(comments)

    save_time = time.perf_counter() - start_save

    print(f"💾 Saved {len(comments)} comments to {filename}")
    print(f"⏱️ CSV save time: {save_time:.2f} seconds")

    return save_time


def main():
    total_start = time.perf_counter()

    comments, fetch_time = get_comments_ytdlp(VIDEO_URL)
    save_time = save_to_csv(comments, OUTPUT_FILE)

    total_time = time.perf_counter() - total_start

    print("\n========== PERFORMANCE ==========")
    print(f"Comments fetched : {len(comments)}")
    print(f"Fetch time       : {fetch_time:.2f} sec")
    print(f"CSV save time    : {save_time:.2f} sec")
    print(f"Total runtime    : {total_time:.2f} sec")
    print("================================")


if __name__ == "__main__":
    main()

✅ Total comments fetched: 10843
⏱️ Fetch time: 131.29 seconds
💾 Saved 10843 comments to comments_ytdlp.csv
⏱️ CSV save time: 0.08 seconds

========== PERFORMANCE ==========
Comments fetched : 10843
Fetch time       : 131.29 sec
CSV save time    : 0.08 sec
Total runtime    : 131.38 sec


# **Method	    Time (seconds)
   Official API	  44.6
   yt-dlp	       131.38
   commntdownloader	226**

In [ ]:
''' Method	    Time (seconds)
   Official API	  44.6
   yt-dlp	       131.38
   commntdownloader	226 '''

' Method\tTime (seconds)\nOfficial API\t44.6\nyt-dlp\t131.38\ncomment-downloader\t226 '

Google rss feed